In [3]:
import pandas as pd
import logging
import requests
import json
import os

# Set up logging
logging.basicConfig(filename='app.log', filemode='w', format='%(asctime)s - %(levelname)s - %(message)s', level=logging.INFO)

def get_publisher_ids(csv_file):
    """
    Read ISSNs from a CSV file, query the CrossRef API, and return a new DataFrame with publisher IDs.
    """

    # Read ISSNs from CSV file
    df = pd.read_csv(csv_file)

    # Extract ISSNs from 2nd column of the csv file
    issns = df.iloc[:, 1].tolist() 

    publisher_ids = []

    # Iterate over the list of ISSNs
    for issn in issns:
        logging.info(f"Processing: {issn}")

        # Define the base URL with ISSN and select parameters
        url = f"https://api.crossref.org/works?filter=issn:{issn}&select=publisher"

        # Make GET request to the CrossRef API
        response = requests.get(url)

        logging.debug(f"API response status code: {response.status_code}")

        # Check for successful response (status code 200)
        if response.status_code == 200:
            logging.info(f"API request successful for {issn}")

            # Parse JSON response
            data = json.loads(response.text)

            # Check if any messages indicating errors
            if "message" in data:
                if "items" in data["message"] and data["message"]["items"]:
                    if len(data["message"]["items"]) > 0:
                        first_item = data["message"]["items"][0]
                        if isinstance(first_item, dict):  # Check if first_item is a dictionary
                            for key, value in first_item.items():
                                publisher_ids.append(value)  # Append the value to the list
                                logging.info(f"Publisher ID found: {value}")
                        else:
                            publisher_ids.append(str(first_item))  # Convert the list to a string and append it
                    else:
                        publisher_ids.append(None)  # Append None if the list is empty
                else:
                    publisher_ids.append(None)  # Append None if "items" key is not present
            else:
                publisher_ids.append(None)  # Append None if "message" key is not present
        else:
            # Handle failed API request
            logging.error(f"Error: API request failed {response.status_code} for {issn}")
            publisher_ids.append(None)  # Append None if the API request fails

    logging.debug("Publisher IDs extracted from API responses")

    # Create a new DataFrame with the original DataFrame adding publisher_ids list
    new_df = pd.DataFrame(list(zip(df.values.tolist(), publisher_ids)), columns=['PMID_issn', 'publisher'])

    logging.info("New DataFrame created:")
    logging.info(new_df)

    logging.debug("Exiting get_publisher_ids function")
    return new_df

# Example usage
csv_file_path = "/Book1.csv"
new_df = get_publisher_ids(csv_file_path)

# Export the new_df to a new CSV file called PMID_Publisher.csv
new_df.to_csv('PMID_Publisher.csv', index=False)
